# Week 2 · Day 5 — 推理评估 + 分割结果可视化

**加载**：`../results/week2_runs/best_model.pt`  
**评估**：全量验证集，逐样本计算 IoU / F1 / AUC

In [ ]:
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
matplotlib.rc('font', family='Microsoft YaHei')
matplotlib.rcParams['axes.unicode_minus'] = False

import sys, yaml
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split
from pathlib import Path
from peft import LoraConfig, get_peft_model, TaskType

MODEL_DIR  = Path('../hls-foundation-os/pretrained_models/prithvi_100m')
PATCH_DIR  = Path('../data/processed/week2_patches')
CKPT_PATH  = Path('../results/week2_runs/best_model.pt')
FIG_DIR    = Path('../results/figures')

sys.path.insert(0, str(MODEL_DIR))
from prithvi_mae import PrithviMAE

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
THRESHOLD = 0.5

MEAN = np.array([343.4, 546.8, 444.1, 2942.5, 1444.6, 899.7], dtype=np.float32)
STD  = np.array([255.0, 340.6, 373.0, 1232.4,  852.0, 680.1], dtype=np.float32)
print(f'设备: {DEVICE}')

## 5.1 重建模型并加载检查点

In [ ]:
class SegDecoder(nn.Module):
    def __init__(self,e=768):
        super().__init__()
        self.up1=nn.Sequential(nn.ConvTranspose2d(e,256,2,2),nn.BatchNorm2d(256),nn.GELU())
        self.up2=nn.Sequential(nn.ConvTranspose2d(256,64,4,4),nn.BatchNorm2d(64),nn.GELU())
        self.up3=nn.Sequential(nn.ConvTranspose2d(64,32,4,4),nn.BatchNorm2d(32),nn.GELU())
        self.head=nn.Conv2d(32,1,1)
    def forward(self,x): return self.head(self.up3(self.up2(self.up1(x))))

class PrithviSegModel(nn.Module):
    NUM_FRAMES=3; EMBED_DIM=768; GRID_SIZE=14
    def __init__(self,prithvi_model):
        super().__init__(); self.prithvi=prithvi_model; self.decoder=SegDecoder()
    def forward(self,x):
        B=x.shape[0]
        x_t=x.unsqueeze(2).repeat(1,1,self.NUM_FRAMES,1,1)
        latent,_,_=self.prithvi.forward_encoder(x_t,mask_ratio=0.0)
        tokens=latent[:,1:,:].reshape(B,self.NUM_FRAMES,self.GRID_SIZE**2,self.EMBED_DIM).mean(1)
        feat=tokens.transpose(1,2).reshape(B,self.EMBED_DIM,self.GRID_SIZE,self.GRID_SIZE)
        return self.decoder(feat)

# 加载 backbone（与 Day 4 完全相同）
sd=torch.load(MODEL_DIR/'Prithvi_100M.pt',map_location='cpu')
prithvi=PrithviMAE(
    img_size=224,patch_size=16,num_frames=3,tubelet_size=1,in_chans=6,
    embed_dim=sd['encoder.norm.weight'].shape[0],depth=12,
    num_heads=sd['encoder.norm.weight'].shape[0]//64,
    decoder_embed_dim=sd['decoder.decoder_embed.bias'].shape[0],
    decoder_depth=8,decoder_num_heads=16,mlp_ratio=4.0,norm_pix_loss=False,
)
prithvi.load_state_dict(sd,strict=False)

model=PrithviSegModel(prithvi)
model.prithvi=get_peft_model(model.prithvi,LoraConfig(
    r=8,lora_alpha=16,lora_dropout=0.1,bias='none',
    target_modules=['qkv','proj'],task_type=TaskType.FEATURE_EXTRACTION
))

# 加载 best_model.pt
ckpt=torch.load(CKPT_PATH,map_location=DEVICE)
model.load_state_dict(ckpt['model_state'])
model=model.to(DEVICE).eval()
print(f'✓ 检查点加载  epoch={ckpt["epoch"]}  val_iou={ckpt["val_iou"]:.4f}')

## 5.2 加载验证集（固定 seed，与训练时一致）

In [ ]:
class ErosionDataset(Dataset):
    def __init__(self,patch_dir):
        self.patch_dir=Path(patch_dir)
        self.indices=sorted(int(p.stem.split('_')[1]) for p in self.patch_dir.glob('img_*.npy'))
    def __len__(self): return len(self.indices)
    def __getitem__(self,idx):
        i=self.indices[idx]
        img =np.load(self.patch_dir/f'img_{i:04d}.npy').astype(np.float32)
        mask=np.load(self.patch_dir/f'mask_{i:04d}.npy').astype(np.float32)
        img=(img-MEAN[:,None,None])/STD[:,None,None]
        return torch.from_numpy(img),torch.from_numpy(mask).unsqueeze(0)

full_ds=ErosionDataset(PATCH_DIR)
n_val=max(1,int(len(full_ds)*.2))
_,val_ds=random_split(full_ds,[len(full_ds)-n_val,n_val],
                      generator=torch.Generator().manual_seed(42))
val_loader=DataLoader(val_ds,batch_size=4,shuffle=False,num_workers=0)
print(f'验证集: {len(val_ds)} 样本')

## 5.3 全量推理

In [ ]:
from torch.utils.data import random_split

all_probs,all_preds,all_masks=[],[],[]
with torch.no_grad():
    for imgs,masks in val_loader:
        probs=torch.sigmoid(model(imgs.to(DEVICE))).cpu()
        all_probs.append(probs)
        all_preds.append((probs>THRESHOLD).float())
        all_masks.append(masks)

all_probs=torch.cat(all_probs); all_preds=torch.cat(all_preds); all_masks=torch.cat(all_masks)

ious,f1s,precs,recs=[],[],[],[]
for i in range(len(all_preds)):
    p=all_preds[i]; g=all_masks[i]
    tp=(p*g).sum(); fp=(p*(1-g)).sum(); fn=((1-p)*g).sum()
    pr=(tp+1e-6)/(tp+fp+1e-6); re=(tp+1e-6)/(tp+fn+1e-6)
    ious.append(((tp+1e-6)/(tp+fp+fn+1e-6)).item())
    precs.append(pr.item()); recs.append(re.item())
    f1s.append((2*pr*re/(pr+re+1e-6)).item())

print('┌' + '─'*42 + '┐')
print('│      Week 2 验证集评估结果              │')
print('├' + '─'*42 + '┤')
for name,vals in [('IoU',ious),('F1',f1s),('Precision',precs),('Recall',recs)]:
    print(f'│  {name:<12}  {np.mean(vals):.4f}  ±{np.std(vals):.4f}         │')
print('└' + '─'*42 + '┘')

## 5.4 指标分布图

In [ ]:
fig,axes=plt.subplots(1,4,figsize=(16,4))
for ax,vals,name,c in zip(axes,[ious,f1s,precs,recs],
    ['IoU','F1','Precision','Recall'],['#e94560','#0f3460','#16213e','#533483']):
    ax.hist(vals,bins=15,color=c,edgecolor='white',lw=0.5)
    ax.axvline(np.mean(vals),color='white',ls='--',lw=1.5,label=f'mean={np.mean(vals):.3f}')
    ax.set_title(name); ax.set_xlim(0,1); ax.legend()
plt.suptitle('验证集指标分布',fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR/'week2_09_metrics_dist.png',dpi=150,bbox_inches='tight')
plt.show()

## 5.5 Best / Middle / Worst 对比

In [ ]:
def denorm(t):
    img=t.numpy()*STD[:,None,None]+MEAN[:,None,None]
    rgb=img[[2,1,0]].copy()
    for i in range(3):
        lo,hi=np.percentile(rgb[i],[2,98])
        rgb[i]=np.clip((rgb[i]-lo)/(hi-lo+1e-8),0,1)
    return rgb.transpose(1,2,0)

all_imgs_list=[full_ds[val_ds.indices[i]][0] for i in range(len(val_ds))]
ious_a=np.array(ious); srt=np.argsort(ious_a); n=min(3,len(srt)//3)
groups=[('Best',srt[-n:][::-1],'#2ecc71'),
        ('Middle',srt[len(srt)//2-n//2:len(srt)//2+n//2+1],'#3498db'),
        ('Worst',srt[:n],'#e74c3c')]
cmap_e=mcolors.ListedColormap(['#1a1a2e','#e94560'])

total_rows=sum(len(g[1]) for g in groups)
fig,axes=plt.subplots(total_rows,4,figsize=(16,4*total_rows))
if total_rows==1: axes=axes[None,:]
row=0
for gname,indices,tcol in groups:
    for idx in indices:
        rgb=denorm(all_imgs_list[idx])
        gt=all_masks[idx,0].numpy(); pred=all_preds[idx,0].numpy(); prob=all_probs[idx,0].numpy()
        axes[row,0].imshow(rgb)
        axes[row,0].set_title(f'{gname}  IoU={ious[idx]:.3f}',color=tcol,fontsize=10,fontweight='bold')
        axes[row,0].axis('off')
        axes[row,1].imshow(gt,cmap=cmap_e,vmin=0,vmax=1); axes[row,1].set_title('Ground Truth'); axes[row,1].axis('off')
        axes[row,2].imshow(pred,cmap=cmap_e,vmin=0,vmax=1); axes[row,2].set_title(f'预测 (τ={THRESHOLD})'); axes[row,2].axis('off')
        im=axes[row,3].imshow(prob,cmap='RdYlGn_r',vmin=0,vmax=1); axes[row,3].set_title('概率热力图'); axes[row,3].axis('off')
        plt.colorbar(im,ax=axes[row,3],fraction=0.046,pad=0.04)
        row+=1

plt.suptitle('分割结果：Best / Middle / Worst',fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR/'week2_09_segmentation_gallery.png',dpi=150,bbox_inches='tight')
plt.show()

## 5.6 ROC 曲线

In [ ]:
from sklearn.metrics import roc_curve, auc

y_true=all_masks.numpy().ravel(); y_prob=all_probs.numpy().ravel()
if len(y_true)>500_000:
    s=np.random.choice(len(y_true),500_000,replace=False)
    y_true=y_true[s]; y_prob=y_prob[s]

fpr,tpr,_=roc_curve(y_true,y_prob)
roc_auc=auc(fpr,tpr)

fig,ax=plt.subplots(figsize=(6,5))
ax.plot(fpr,tpr,color='#e94560',lw=2,label=f'ROC (AUC={roc_auc:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1,label='随机猜测')
ax.set_xlabel('FPR'); ax.set_ylabel('TPR'); ax.set_title('ROC 曲线'); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout()
plt.savefig(FIG_DIR/'week2_09_roc_curve.png',dpi=150,bbox_inches='tight')
plt.show()
print(f'AUC = {roc_auc:.4f}')

## 5.7 Week 2 总结

In [ ]:
total_p  = sum(p.numel() for p in model.parameters())
train_p  = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('\n' + '='*55)
print('  Week 2 完成总结')
print('='*55)
print(f'  Notebook  │ 内容')
print(f'  ──────────┼──────────────────────────────────────')
print(f'  05        │ 光谱指数 + 从 3660×3660 大图采样 patch')
print(f'  06        │ ErosionDataset + 类别不平衡分析')
print(f'  07        │ PrithviMAE + LoRA(r=8) + SegDecoder')
print(f'  08        │ 30 epoch 训练 + TensorBoard')
print(f'  09        │ 全量评估 + Best/Worst 可视化')
print('─'*55)
print(f'  模型        PrithviMAE + LoRA(r=8) + SegDecoder')
print(f'  输入格式    (B,6,3,224,224)  T=3 复制')
print(f'  Token处理   588 tokens → 时间均值 → (B,768,14,14)')
print(f'  总参数      {total_p/1e6:.1f}M')
print(f'  可训练参数  {train_p/1e6:.2f}M ({100*train_p/total_p:.1f}%)')
print('─'*55)
print(f'  Val IoU    {np.mean(ious):.4f}')
print(f'  Val F1     {np.mean(f1s):.4f}')
print(f'  AUC        {roc_auc:.4f}')
print('─'*55)
print('  Week 3 方向')
print('    • 多时相真实输入（T=3 不同日期）')
print('    • 真实标注标签替换伪标签')
print('    • FPN / UPerNet 解码头')
print('='*55)

---
## 💾 Week 2 全部成果推送 GitHub

In [ ]:
import subprocess, os
REPO_DIR=str(Path('..').resolve()); os.chdir(REPO_DIR)
def git(cmd):
    r=subprocess.run(cmd,shell=True,capture_output=True,text=True,cwd=REPO_DIR)
    out=(r.stdout+r.stderr).strip()
    if out: print(out)

print('── git status ──')
git('git status --short')

In [ ]:
# Week 2 全部 Notebooks
for nb in ['05_spectral_labels','06_erosion_dataset','07_lora_model','08_training','09_evaluation_vis']:
    git(f'git add notebooks/{nb}.ipynb')

# 可视化结果图
for fig_name in [
    'week2_05_spectral_indices','week2_05_label_check',
    'week2_06_class_balance','week2_06_batch_preview',
    'week2_08_training_curves',
    'week2_09_metrics_dist','week2_09_segmentation_gallery','week2_09_roc_curve'
]:
    git(f'git add results/figures/{fig_name}.png')

msg = (
    f'Week2 complete: PrithviMAE + LoRA(r=8) soil erosion seg | '
    f'val_iou={np.mean(ious):.4f} f1={np.mean(f1s):.4f} auc={roc_auc:.4f}'
)
git(f'git commit -m "{msg}"')

In [ ]:
git('git push origin main')
print()
print('✓ Week 2 全部成果已推送:')
print('  https://github.com/lofophil/geo-foundation-experiments')
print('  · 5 个 Notebooks (05~09)')
print('  · 8 张可视化结果图')
print(f'  · Commit 含关键指标 IoU={np.mean(ious):.4f}')